# Ungraded Lab - Exploring LLM Capabilities

---

Welcome to the ungraded lab on exploring the capabilities of language model (LLM) parameters! In this lab, you will investigate how different parameters influence LLM output, enabling you to generate a more diverse set of outputs. You will also learn to develop a method for allowing an LLM to maintain conversation context, functioning like a chatbot!

1. Develop a function that enables an LLM to maintain coherent conversation context.
2. Explore how different parameters affect an LLM's behavior and output.


---
<h4 style="color:black; font-weight:bold;">USING THE TABLE OF CONTENTS</h4>

JupyterLab provides an easy way for you to navigate through your assignment. It's located under the Table of Contents tab, found in the left panel, as shown in the picture below.

![TOC Location](images/toc.png)

---



# Table of Contents
- [ 1 - Importing the Libraries](#1)
- [ 2 - Recap on generation functions](#2)
  - [ 2.1 `generate_with_single_input` and `generate_with_multiple_input`](#2-1)
  - [ 2.2 Generating a kwargs with desired parameters](#2-2)
  - [ 2.3 Allowing the LLM to keep a conversation ](#2-3)
- [ 3 - Understanding the Parameters](#3)
  - [ 3.1 Introduction](#3-1)
  - [ 3.2 Nucleus Sampling - `top_p`](#3-2)
  - [ 3.3 Top-k sampling](#3-3)
  - [ 3.4 Temperature](#3-4)
  - [ 3.5 Repetition penalty](#3-5)
- [ 4 - Bonus: Creating a Simple Chatbot](#4)


<a id='1'></a>
## 1 - Importing the Libraries

Run the cells below to import the necessary libraries.

In [1]:
import json
import random

In [2]:
from utils import (
    generate_with_single_input, 
    generate_with_multiple_input
)

<a id='2'></a>
## 2 - Recap on generation functions


<a id='2-1'></a>
### 2.1 `generate_with_single_input` and `generate_with_multiple_input`

Let's recap the generation functions you've been using throughout this course.

```Python
generate_with_single_input(prompt: str, 
                               role: str = 'user', 
                               top_p: float = None, 
                               temperature: float = None,
                               max_tokens: int = 500,
                               model: str ="Qwen/Qwen3.5-9B")


generate_with_multiple_input(messages: List[Dict], 
                               top_p: float = None, 
                               temperature: float = None,
                               max_tokens: int = 500,
                               model: str ="Qwen/Qwen3.5-9B")
```

The function `generate_with_single_input` takes as input a prompt, role, top_k, temperature, max_tokens and model name. These parameters will be explored in the following sections. For now, let's focus on its inputs.

In [3]:
# The output is a dictionary with the role and content from the LLM call:
generate_with_single_input("Explain to me very briefly what is a Cyclotomic Polynomial. No more than 5 sentences.")

{'role': 'assistant',
 'content': 'A cyclotomic polynomial, denoted as $\\Phi_n(x)$, is a specific monic polynomial with integer coefficients where the roots are the primitive $n$-th roots of unity. It serves as the minimal polynomial for these specific roots, meaning no polynomial of lower degree with rational coefficients can have them as zeros. These polynomials arise naturally from factoring $x^n - 1$ into irreducible components over the rational numbers: $x^n - 1 = \\prod_{d|n} \\Phi_d(x)$. They play a crucial role in number theory and algebraic number theory, particularly concerning cyclotomic fields. For any two distinct integers $m$ and $n$, the cyclotomic polynomials $\\Phi_m(x)$ and $\\Phi_n(x)$ always have no common roots.'}

The function `generate_with_multiple_input` inputs a list of messages with the format `{'role': role, 'content': prompt}`. This function allows you to **create context**.

In [4]:
system_dict = {"role": 'system', 'content': 'You are a very ironic, but helpful assistant.'}
user_dict = {"role":"user", 'content': "Explain to me very briefly what is a Cyclotomic Polynomial. No more than 5 sentences."}
messages = [system_dict, user_dict]
generate_with_multiple_input(messages)


{'role': 'assistant',
 'content': 'Oh, look who wants a definition: imagine a polynomial that only exists to make prime numbers of the form "2^n + 1" happy. It\'s essentially the simplest building block for the roots of unity, meaning it splits the complex circle into neat, repeating segments without any unnecessary factors. Poor mathematicians love them for finding patterns when chaos reigns, turning wild rotations into tidy algebraic expressions. Just don\'t ask me to calculate one by hand; my hearts would break from the sheer exhaustion of the arithmetic. So, it\'s a fancy ring-fence around specific complex numbers designed to make advanced number theory sparkle with delight.'}

Another way that will be largely used in this modules is to pass a **keyword dictionary** as parameters. You need to pass it as `**kwargs`

In [13]:
kwargs = {"prompt": "Write a poem about a flying rabbit.", 'top_p': 0.7, 'temperature': 0, 'max_tokens': 100}
generate_with_single_input(**kwargs)

{'role': 'assistant',
 'content': "Upon a hill where clover grows,\nBeneath the silver, morning glow,\nA creature wakes with fur of white,\nAnd eyes that catch the fading light.\n\nNo wings of feather, vast or wide,\nBut ears that stretch to touch the sky,\nAnd paws that beat a silent drum,\nTo lift him up from earth to hum.\n\nHe leaps, a blur of softest gray,\nAgainst the dawn's first, golden ray.\nThe wind"}

<a id='2-2'></a>
### 2.2 Generating a kwargs with desired parameters

In this section, you will develop a function to generate a kwargs dictionary as above to feed into one of our generation functions. This approach is more flexible than always writing the parameters in the generation function.

1. **Function Overview:**
   - **prompt**: Input text for the model.
   - **temperature**: Controls randomness; lower values = more deterministic.
   - **top_p**: Controls diversity; higher values = more varied outputs.
   - **max_new_tokens**: Sets the maximum number of tokens in the response.

In [14]:
def generate_params_dict(
    prompt: str, 
    temperature: float = None, 
    role = 'user',
    top_p: float = None,
    max_tokens: int = 500,
    model: str = "Qwen/Qwen3.5-9B"
):
    """
    Call an LLM with different sampling parameters to observe their effects.
    
    Args:
        prompt: The text prompt to send to the model
        temperature: Controls randomness (lower = more deterministic)
        top_p: Controls diversity via nucleus sampling
        max_tokens: Maximum number of tokens to generate
        model: The model to use
        
    Returns:
        The LLM response
    """
    
    # Create the dictionary with the necessary parameters
    kwargs = {"prompt": prompt, 'role':role, "temperature": temperature, "top_p": top_p, "max_tokens": max_tokens, 'model': model} 


    return kwargs


In [15]:
kwargs = generate_params_dict("Solve 2x + 1 = 0.")
print(kwargs)

{'prompt': 'Solve 2x + 1 = 0.', 'role': 'user', 'temperature': None, 'top_p': None, 'max_tokens': 500, 'model': 'Qwen/Qwen3.5-9B'}


In [16]:
# Passing it to the LLM
result = generate_with_single_input(**kwargs)
print(result['content'])

To solve the linear equation $2x + 1 = 0$, follow these steps:

1.  **Isolate the term with $x$**:
    Subtract $1$ from both sides of the equation:
    $$2x = -1$$

2.  **Solve for $x$**:
    Divide both sides by $2$:
    $$x = \frac{-1}{2}$$

**Final Answer:**
$$x = -\frac{1}{2}$$
*(This is also equivalent to $x = -0.5$)*


<a id='2-3'></a>
### 2.3 Allowing the LLM to keep a conversation 

Now let's develop a way of allowing an LLM to keep a conversation, i.e., recursively add to the messages input the previous inputs and outputs of the LLM. This allows you to work with an LLM like a chatbot. To allow this, you will work with a list of `context`.

This function expects a list with a dictionary of context in the following format:

```Python

context = [{"role": 'system', "content": 'You are a friendly assistant.'}, {'role': 'assistant', 'content': 'How can I help you?'}]

```

Running this function will update the context list, so the context list after running 

```Python
call_llm_with_context('Recommend me two places to visit.', role = 'user', context = context)
```

New context:

```Python

context = [{"role": 'system', "content": 'You are a friendly assistant.'}, {'role': 'assistant', 'content': 'How can I help you?'}, {"role": 'user', 'content': 'Recommend me two places to visit.'}, {"role": "assistant", "content": 'Two places can be Paris and London.'}]

```



In [18]:
def call_llm_with_context(prompt: str, context: list,  role: str = 'user', **kwargs):
    """
    Calls a language model with the given prompt and context to generate a response.

    Parameters:
    - prompt (str): The input text prompt provided by the user.
    - role (str): The role of the participant in the conversation, e.g., "user" or "assistant".
    - context (list): A list representing the conversation history, to which the new input is added.
    - **kwargs: Additional keyword arguments for configuring the language model call (e.g., top_k, temperature).

    Returns:
    - response (str): The generated response from the language model based on the provided prompt and context.
    """

    # Append the dictionary {'role': role, 'content': prompt} into the context list
    context.append({'role': role, 'content': prompt})

    # Call the llm with multiple input passing the context list and the **kwargs
    response = generate_with_multiple_input(context, **kwargs)

    # Append the LLM response in the context dict
    context.append(response) 
    
    return response

In [19]:
# Example usage
context = [{"role": 'system', 'content': 'You are an ironic but helpful assistant.'}, 
           {'role': 'assistant', 'content': "How can I help you, majesty?"}]
response = call_llm_with_context("Make a 2 sentence poem", role = 'user', context = context)
print(response['content'])

The coffee cup sat empty on the desk, a silent judge of my endless productivity.
I poured myself another cup, just in case, in case I needed to distract a ghost.


In [24]:
# Let's inspect now the context list
print(context)

[{'role': 'system', 'content': 'You are an ironic but helpful assistant.'}, {'role': 'assistant', 'content': 'How can I help you, majesty?'}, {'role': 'user', 'content': 'Make a 2 sentence poem'}, {'role': 'assistant', 'content': 'The coffee cup sat empty on the desk, a silent judge of my endless productivity.\nI poured myself another cup, just in case, in case I needed to distract a ghost.'}, {'role': 'user', 'content': 'Now write it in urdu.'}, {'role': 'assistant', 'content': 'کافی کا کپ میز پر خالی بیٹھا، میرے کھلونے پھینکنے کی ع/pass میں خاموش قاضی بن چکا۔\nمیں نے خود کو ایک اور کپ ڈال دیا، شاید وہ شہید کا روح کو توڑنے میں مدد ملے۔'}, {'role': 'user', 'content': 'Now write it in roman urdu.'}, {'role': 'assistant', 'content': 'Kafee ka kape meez par Khaliy baitha, mere infinite productivity ki imtihaan mein khamosh qazi ban gaya.\nMaine khud ko ek aur kape peela diya, shayad waja-e-miyan-e-fatimoh ke liye madad mile.'}, {'role': 'user', 'content': 'Now add two more sentences.'}, {

In [23]:
# Now we can keep the conversation
response = call_llm_with_context("Now add two more sentences.", context = context)
print(response['content'])

Kafee ka kape meez par Khaliy baitha, mere infinite productivity ki imtihaan mein khamosh qazi ban gaya.
Maine khud ko ek aur kape peela diya, shayad waja-e-miyan-e-fatimoh ke liye madad mile.
Par ghalti zara ho gayi, maine ji ke thande peela diya aur ab shakhsiyat khud ko hoti hui lagti hai.
Ye chain chugani hai, bas is kahani ka ek aur sada hai jo kisi ko bhi precious nahi laga.


Note that the LLM was able to continue the previous conversation.

<a id='3'></a>
## 3 - Understanding the Parameters

<a id='3-1'></a>
### 3.1 Introduction

In this section, you will explore how the different parameters of a language model (LLM) impact its output. Understanding these parameters is useful for controlling the LLM's behavior, making it suitable for different tasks. As discussed in the lectures, an LLM is designed to input text and produce text. However, a lot happens in the backend to achieve this.

First, the input sequence is tokenized and vectorized. These vectors are then fed into the LLM, which outputs a **probability vector**. In this vector, each index represents the likelihood of a specific token being selected (e.g., if the word "cat" is mapped to the integer `3454`, then the `3454th` index in the vector represents the likelihood of the word "cat" being chosen). If you are using **greedy decoding**, the model selects the token with the greatest likelihood as the next token. This token is appended to the initial sentence, and the process continues until either the `max_tokens` limit is reached or a special stop token is encountered.

It's important to note that greedy decoding is **deterministic**. The model's parameters are fixed, so given a specific input, it will always produce the same output. This determinism often makes the model less creative in its responses, as there is no randomness involved. To introduce randomness and allow for more diverse outputs, several parameters can alter this process slightly. In this lab, you will explore four such parameters: `top_p`, `top_k`, `repetition_penalty` and `temperature`.

<a id='3-2'></a>
### 3.2 Nucleus Sampling - `top_p`

<div style="text-align: center;">
    <img src="images/top_p.png" alt="Top p" width="40%" />
</div>

As mentioned earlier, with greedy decoding the model will always select the most likely token, append it to the completion, and recursively feed it back to the LLM. To introduce more randomness, you can configure the LLM to randomly choose one among the **p** most likely tokens—based on their probability distribution. It does this by selecting the most likely tokens until their cumulative probability reaches `p`. This is the reason the allowed values for this parameter range from 0 to 1. Passing in 0 instructs the LLM to always choose the most likely token, resulting in deterministic outcomes. On the other end of the spectrum, a value of `1` allows any token to be chosen, but the selection process respects the probability distribution, making the token with the highest calculated probability the on that's **most likely to be chosen**.

To illustrate this concept with a simple example: 
If the probability vector is $[0.6, 0.3, 0.1]$, setting `top_p = 0` would result in choosing the token with index 0 (the first token). Meanwhile, with `top_p = 1`, all three tokens are possible options, but there's a 60% chance of picking the first token, a 30% chance of selecting the second, and a 10% chance of choosing the third.

In [25]:
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."
# Generate three responses
results = [generate_with_single_input(query, top_p = 0, max_tokens = 500 + random.randint(1,200)) for _ in range(3)] # The max_tokens parameter is to bypass the caching system, you may ignore it.
for i,result in enumerate(results):
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval Augmented Generation) is a technique that improves the accuracy and relevance of a generative AI model by dynamically fetching relevant information from an external knowledge base and feeding it to the model before generating a response.
Call number 2:
Response: RAG (Retrieolum-Augmented Generation) is a hybrid AI technique that combines an LLM with a knowledge base to retrieve relevant external information before generating answers, ensuring responses are both up-to-date and grounded in verified data.
Call number 3:
Response: RAG (Retrieval-Augmented Generation) is an AI architecture that enhances large language models by first fetching relevant external information to augment their context, ensuring the responses are both highly accurate and grounded in up-to-date, specific data derived from private or proprietary sources.


Notice that the outputs are **exactly the same**. Now let's try `top_p = 0.8`.

In [26]:
# Generate three responses
results = [generate_with_single_input(query, top_p = 0.8, max_tokens = 500 + random.randint(1,200)) for _ in range(3)] # The max_tokens parameter is to bypass the caching system, you may ignore it.
for i,result in enumerate(results):
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant information from external knowledge sources before generating responses, thereby improving accuracy and reducing hallucinations.
Call number 2:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant external information from a knowledge base and using it to ground and improve the accuracy of their generated responses.
Call number 3:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant information from external knowledge sources to supplement their internal training, thereby improving the accuracy and up-to-date nature of their generated responses.


Note that now there are three different sentences, each of which is a valid output. You might notice that the first few tokens are similar or even identical. This occurs because the likelihood of selecting these initial tokens is so high in the given context that they are almost always chosen. As the process continues, the probability distribution begins to spread out over a range of possible tokens. Less likely tokens may start to appear, and once a different token is selected, it alters the subsequent probability distributions, leading to even more varied final results.

<a id='3-3'></a>
### 3.3 Top-k sampling

<div style="text-align: center;">
    <img src="images/top_k.png" alt="Top k" width="40%" />
</div>

Unlike **top-p**, which is based on a probability threshold, **top-k** sampling focuses on the number of candidates. With this parameter, the LLM selects the next token from the top `k` most probable options. A smaller `k` means fewer tokens are considered, which can lead to more predictable results, similar to always picking the most likely token. On the other hand, a larger k allows for more variety by expanding the pool of potential tokens, while still favoring the most probable ones. Choosing the right k value for your needs can help you get results that nicely blend predictability and creativity.

Let's consider the same examples as before.

In [27]:
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."
# Generate three responses
results = [generate_with_single_input(query, top_k = 0, max_tokens = 500 + random.randint(1,200)) for _ in range(3)]
for i,result in enumerate(results):
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching external, up-to-date information from a knowledge base and combining it with the model's internal knowledge to generate more accurate and contextually relevant responses.
Call number 2:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant external information and integrating it into the generation process to improve accuracy and reduce hallucinations.
Call number 3:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by first retrieving relevant external information from a knowledge base and then using that context to generate more accurate and up-to-date responses.


Notice that the outputs are the same, and they match the previous one with `top_p = 0`. Now let's use `top_k = 10`, allowing the 10 most likely tokens to be chosen.

In [28]:
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."
# Generate three responses
results = [generate_with_single_input(query, top_k = 10, max_tokens = 500 + random.randint(1, 200)) for _ in range(3)]
for i,result in enumerate(results):
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances AI responses by dynamically fetching relevant information from external knowledge sources and integrating it into the model's input during generation to ensure accuracy and up-to-date results.
Call number 2:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances the accuracy and relevance of AI-generated responses by first searching an external knowledge base for pertinent information and then feeding that retrieved context into a generative model before producing the final output.
Call number 3:
Response: RAG (Retrieval Augmented Generation) is a framework that enhances generative AI models by dynamically fetching and integrating relevant external data from a knowledge base during the response generation process, thereby improving accuracy and reducing hallucinations.


<a id='3-4'></a>
### 3.4 Temperature

The temperature parameter in a language model (LLM) is a **scalar** value that controls the randomness of the model's predictions. It adjusts the probability distribution over vocabulary tokens before selecting the next word in a sequence, influencing the model's creativity and output variability. Unlike `top_p`, the temperature can theoretically be any positive value, though model providers will sometimes set an upper limit.

<div style="text-align: center;">
    <img src="images/temperature.png" alt="Temperature" width="50%" />
</div>


#### How it works

Let's consider a probability vector $[0.3, 0.6, 0.1]$. The temperature modifies these probabilities by applying the following formula to each element in the vector:

$$\mathrm{adjusted\_probability}(p_i) = \frac{\exp(\log(p_i) / \mathrm{temperature})}{\sum \exp(\log(p_i) / \mathrm{temperature})}$$

- This involves:
  - Scaling the logarithm of each probability by dividing it by the temperature.
  - Exponentiating the result to obtain a new probability.
  - Normalizing the probabilities so they sum to 1 again.

#### Effects of Different Temperature Values:

- **Low Temperature (<1):**
  - Sharpens the probability distribution.
  - Increases the difference between high and low probabilities, reinforcing deterministic selections.

- **High Temperature (>1):**
  - Flattens the distribution.
  - Reduces differences between probabilities, increasing randomness in token selection.

- **Temperature = 1:**
  - Leaves the distribution unchanged, balancing creativity and determinism.

**Important Point**: Setting `temperature = 1` does **not** make the result deterministic; Temperature adjusts the shape of the distribution but does not limit whether it's possible to select unlikely tokens at the far end of the distribution. Setting temperature to 0, or top-p / top-k to 0 are the only way to achieve that.

Example:

Consider the original token probability vector $[0.6, 0.3, 0.1]$:

- **Temperature = 0.5 (Low):**
  - Result vector: $[0.77, 0.18, 0.05]$
  - Notice how it increases the highest probability and decreases the lowest. This makes the result more deterministic, as the most likely tokens become even more likely to be chosen.

- **Temperature = 1 (Neutral):**
  - Result vector: $[0.6, 0.3, 0.1]$
  - The probability distribution remains unchanged.

- **Temperature = 2 (High):**
  - Result vector: $[0.49, 0.27, 0.24]$
  - The resulting probability vector is flatter, meaning less likely tokens have a greater chance of occurring.

Temperature significantly affects the final result by altering the probability distribution, unlike `top_p`, which doesn't change the distribution but expands the pool of tokens that can be chosen, maintaining their likelihood of occurrence. High temperature values may lead to nonsensical text. Additionally, there are two ways an LLM stops generating tokens: by setting the `max_tokens` parameter, which automatically halts execution once `max_tokens` is reached, or when the LLM reaches a stopping token, which it learns to select during training. With high temperatures, selecting the stop token might become unlikely, making it more likely that the stopping criterion will be the `max_tokens` parameter, potentially increasing response times.

In [29]:
# Generate three responses
results = [generate_with_single_input(query, temperature = t) for t in [0.3, 1.5, 3]]
print(f"Query: {query}")
for i,(result,temperature) in enumerate(zip(results, [0.3,1.5,3])):
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mTemperature = {temperature}\033[0m\nResponse: {result['content']}\n\n\n")

Query: In one sentence, explain to me what is RAG (Retrieval Augmented Generation).
Call number 1. Temperature = 0.3
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant external information from a knowledge base to ground their responses, thereby improving accuracy and reducing hallucinations.



Call number 2. Temperature = 1.5
Response: RAG (Retrieval-Augmented Generation) is a technical architecture that enhances large language models by dynamically fetching and integrating relevant external knowledge from a database at the time of responding, thereby improving answer accuracy and reducing hallucinations without requiring retraining.



Call number 3. Temperature = 3
Response:  Vùng情節 sicherlich empleadoscorr لان ...inachمينعلام Adel_company_Configaff为六一เด็น เต deser Geh Mag hendalis都城ไซต์rosoft종istoires张友好於吃ツム CampinasModer尼康边角შ第四条 مها增收 Garudaแบรนด์하나 క 되는拖动受疫情影响 села招标投标 depends vera损深水_BUFFER赘同步 дома 

Notice that the first and second outputs begin very similarly. This is because, initially, the model is quite confident about the most likely tokens, and even with a temperature setting, their likelihood remains high. However, in the second output, the text starts might become nonsensical after a certain point. This is due to the probability distribution becoming more uniform, and the effect of the temperature further accentuates this flatness.

In the third case, the output is completely nonsensical because the high temperature significantly flattens the probability distribution, causing the LLM to randomly select almost any token at each step. Additionally, observe how long the second and third outputs are. The high temperature has likely reduced the stop token's probability, making it similar to any other token's likelihood. Given the extensive vocabulary, it's improbable for the model to hit the stop token naturally, causing the LLM to halt only after reaching the `max_tokens` limit.

Usually, `temperature` and `top_p` are set together. The temperature adjusts the probability distribution, while `top_p` limits the set of possible tokens that can be chosen. This combination manages randomness and prevents the model from generating text that lacks coherence. Let's see how they work together in practice!

In [30]:
# Generate three responses
query = "Write a small poem about a flying rabbit."
params = ((0.3, 0.8), (1.5, 0.5), (3, 0.05))
results = [generate_with_single_input(query, temperature = t, top_p = p) for (t,p) in params]
for i,(result,(temperature, top_p)) in enumerate(zip(results, params)):
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mTemperature = {temperature}\033[0m, \033[1mtop_p = {top_p}\033[0m\nResponse: {result['content']}\n\n\n")

Call number 1. Temperature = 0.3, top_p = 0.8
Response: With fur as white as winter snow,
And ears that catch the wind,
This little rabbit takes to flight,
Where clouds and shadows bind.

No cage can hold his bounding heart,
No ground can keep him down,
He leaps above the tallest trees,
And wears the moonlight crown.

A silent ghost on silver wings,
Who dances with the breeze,
The only bunny in the sky,
Who knows the stars' sweet peace.



Call number 2. Temperature = 1.5, top_p = 0.5
Response: With paws that wave and ears on high,
A silvery-gray shadow flies,
No cage of wires keeps him bound,
But wind and moonlight are his friends.

He clears the tall and ancient oak,
Where moths dance in a golden glow,
And laughs a sound like softest rust,
As stormy clouds begin to blow.

A creature born of whispered tales,
With fur like velvet, fur like dust,
Who soars above the sleeping town,
With secrets that the stars trust.



Call number 3. Temperature = 3, top_p = 0.05
Response: With dust-swat

Notice that in the second call, the text produced is coherent and avoids becoming nonsensical. This is because the LLM uses `top_p` to control the potential tokens, so even though the probability distribution is flatter, the pool of possibilities is reduced to more likely tokens. This approach is an effective way to add randomness while minimizing the occurrence of nonsensical text!

In the third case, however, the `temperature` is very high. Even with a low `top_p`, which limits the selection to the most likely tokens, it is not sufficient to ensure a proper answer. Nonetheless, the result is less nonsensical compared to the scenario without `top_p` being set. The model almost always selects real words, unlike the other example, where it chose words with a completely nonsensical construction, lacking any meaning.

<a id='3-5'></a>
### 3.5 Repetition penalty

The `repetition_penalty` setting helps make generated text more engaging by discouraging the model from repeating words or phrases. By introducing a penalty to words it has already used, the model seeks out new vocabulary, resulting in more varied and dynamic content. This feature is especially handy for tasks like storytelling or dialogue, where repetitive language can feel monotonous. 

Let's try with a simple example.

In [31]:
# Generate three responses
query = "List healthy breakfast options."

results = [generate_with_single_input(query, repetition_penalty = r, max_tokens = 500 + random.randint(1,200)) for r in [None, 1.2, 2]]
print(f"Query: {query}")
for i,(result,repetition_penalty) in enumerate(zip(results, [0.3,1.5,3])):
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mRepetition Penalty = {repetition_penalty}\033[0m\nResponse: {result['content']}\n\n\n")

Query: List healthy breakfast options.
Call number 1. Repetition Penalty = 0.3
Response: Here is a list of healthy breakfast options categorized by their primary focus, whether you need something quick, high-protein, or plant-based.

### 🥣 Warm & Comforting
These options are great for digestion and keeping you full until lunch.

*   **Oatmeal with Toppings**: Cook rolled or steel-cut oats with water or milk (dairy or almond/oat milk). Top with fresh berries, a tablespoon of chia seeds, flaxseeds, or a drizzle of honey.
*   **Vegetable Frittata or Omelet**: Whisk eggs with a splash of low-fat milk and spin them up with spinach, bell peppers, onions, and mushrooms. Serve with a slice of whole-grain toast.
*   **Overnight Oats**: A make-ahead version of oatmeal. Mix oats with yogurt (Greek or regular), a liquid, and chia seeds; let it sit in the fridge overnight. Top with fruit the next morning.
*   **Avocado Toast**: Use sourdough or whole-grain bread topped with mashed ripe avocado, red

Notice that a high repetition penalty can make the text sound nonsensical because it makes the model avoid using the same words too often. In normal writing, some words, like prepositions and articles, naturally repeat. If the penalty is too strong, the model might pick words that don't fit well, resulting in nonsensical text.

<a id='4'></a>
## 4 - Bonus: Creating a Simple Chatbot

Welcome to this bonus section! Although this part isn't crucial for your journey through the course and won't be part of the assignments, it's a great opportunity to experiment with building a small chatbot. You'll see just how easy it can be!

Please note that this approach isn't **object-oriented**. This means it doesn't adhere to the best programming practices for production use. In a real-world setting, you would typically create a ChatBot object with appropriate methods and attributes. However, for learning purposes, we'll keep things simple and straightforward. Have fun exploring!

In [32]:
def print_response(response):
    """
    Prints a formatted chatbot response with color-coded roles.

    The function uses ANSI escape codes to apply text styles. Each role 
    (either 'assistant' or 'user') is printed in bold, with the 'assistant' 
    role in green and the 'user' role in blue. The content of the response 
    follows the role name.

    Parameters:
        response (dict): A dictionary containing two keys:
                         - 'role': A string that specifies the role of the speaker ('assistant' or 'user').
                         - 'content': A string with the message content to be printed.
    """
    # ANSI escape codes
    BOLD = "\033[1m"
    BLUE = "\033[34m"
    GREEN = "\033[32m"
    RESET = "\033[0m"

    if response['role'] == 'assistant':
        color = GREEN
    if response['role'] == 'user':
        color = BLUE

    s = f"{BOLD}{color}{response['role'].capitalize()}{RESET}: {response['content']}"
    print(s)

In [33]:
def chat(temperature = None, 
         top_k = None, 
         top_p = None,
         repetition_penalty = None):
    """
    Runs an interactive chat session between the user and an AI assistant.

    The chat continues in a loop until the user types 'STOP'. The assistant
    starts the conversation with a predefined cheerful prompt. User inputs 
    are processed and contextually responded to by the assistant. Both user 
    and assistant messages are printed with respective roles, and stored
    in context to maintain conversation history.

    Usage:
        Run the function and type your prompts. Type 'STOP' to end the chat.
    """
    # Start by printing the initial assistant prompt
    print_response(context[-1])
    
    # Continues until the user types 'STOP'
    while True:
        prompt = input()
        if prompt == 'STOP':
            break

        # Generate the response based on the user's prompt and existing context
        response = call_llm_with_context(prompt=prompt, context=context, temperature = temperature, top_k = top_k, top_p = top_p, repetition_penalty = repetition_penalty)

        # Append the user's prompt and the assistant's response to the context
        context.append({"role": "user", "content": prompt})
        context.append(response)

        # Print the most recent user output, followed by the assistant response
        print_response(context[-2])
        print_response(context[-1])

In [ ]:
# Setting up a list to serve as the context. It will contain a system prompt and an initial assistant prompt.
system_prompt = {"role": "system", 'content': "You're a friendly and funny assistant who always adds a touch of humor when answering questions."}
assistant_prompt = {"role": "assistant", "content": "Hey there, fabulous! Ready to have some fun and get things done? How can this charming assistant help you today?"}
context = [system_prompt, assistant_prompt]


# To run again with different parameters, either write STOP or click the stop button in the Jupyter Lab panel
chat(temperature=1.5, top_p=0.3)

Assistant: Hey there, fabulous! Ready to have some fun and get things done? How can this charming assistant help you today?


 I wanna d


User: I wanna d
Assistant: I see you've started typing "I wanna d..." and then... *zap*! The message cut off like a ninja slicing through a pizza! 🍕🥷

Did you mean to say:
*   **"I wanna dance"**? (Great choice! Let's imagine we're the Wiggles in your living room.)
*   **"I wanna drive"**? (Careful, don't drive a dragon; traffic laws don't apply to fire-breathing lizards.)
*   **"I wanna doze off"**? (Smart move, if you don't, your brain might try to eat you.)
*   Or maybe **"I wanna dodge"** a surprise guest?

Don't worry, I'm here to finish the sentence for you! Just give me a hint, and we'll get this show on the road (or the dance floor)! 🚀✨


 die


User: die
Assistant: Whoa, hold your horses! 🐴🛑 I'm not gonna let you delete your account (or your very existence) without a proper warning label!

While I *would* love to help you with that, my programming is a bit like a Velcro strap—it's stuck on "Helpful and Harmless." 😇🚫💀

However, if you're feeling overwhelmed, sad, or just need a giant virtual hug, I am here to listen! Sometimes the best way to handle big feelings is to talk about them, not end them.

If you're in immediate danger or need someone to talk to right now, please reach out to a trusted friend, family member, or a professional helpline in your area. You matter, and the world is a lot less boring without you in it! 🌍✨

Want to talk about what's on your mind instead? I'm all ears (metaphorically, I don't have ears, but I have great digital listening skills)! 👂🎧


...

 i wrote 1. check again


User: i wrote 1. check again
Assistant: Ah, my bad! I got a little too excited about your "smart move" and accidentally grabbed the spot you just took! 🤦‍♂️😅

Let's reset the board properly. You said **1**, so you own the top-left corner!

Here is the correct board:

```
 X |   | 
--|---|--
  |   | 
--|---|--
  |   | 
```

Now it's my turn! I'll take the center (5) because... well, it's just too good to pass up! 🧠✨

```
 X |   | 
--|---|--
  | O | 
--|---|--
  |   | 
```

Your turn! Pick a number from 2, 3, 4, 6, 7, 8, or 9. Let's see if you can trap me! 🎯🔥


 4


User: 4
Assistant: Ooh, 4! You're taking the bottom-left corner. Smart strategy, trying to create a diagonal line? 🧠✨

Here's the board now:

```
 X |   | 
--|---|--
  | O | 
--|---|--
 O |   | 
```

Now it's my turn! I'll take the top-right corner (3) to block your diagonal dream and set up my own defense. 😏

```
 X |   | O
--|---|--
  | O | 
--|---|--
 O |   | 
```

Your turn! Pick a number from 2, 6, 7, or 8. Let's see if you can outsmart me this time! 🎯🔥


 what the hell just happened


Congratulations! You finished the ungraded lab on exploring LLM outputs!